# 09 — StockGro Trading Signals

Generate trading instructions and StockGro formatted outputs using model forecasts and portfolio weights.

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
PORT_DIR = ROOT / 'outputs' / 'portfolio'
REPORT_DIR = ROOT / 'outputs' / 'reports'
PRED_DIR = ROOT / 'outputs' / 'predictions'

import pandas as pd
import numpy as np
from src.data.preprocessor import SELECTED, NAMES
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

# ── Load portfolio allocation ─────────────────────────────────
port_file = PORT_DIR / 'final_portfolio_allocation.csv'
if not port_file.exists():
    print('ERROR: Portfolio file not found. Run notebook 08 first.')
else:
    portfolio = pd.read_csv(port_file)
    print(f'✅ Loaded portfolio with {len(portfolio)} stocks')
    
    # ── Load ensemble 5-day forecasts ─────────────────────────
    print('Loading ensemble forecasts...')
    
    forecast_data = {}
    forecast_files = [
        PRED_DIR / 'arima_5day_forecasts.csv',
        PRED_DIR / 'ets_5day_forecasts.csv',
        PRED_DIR / 'prophet_5day_forecasts.csv',
        PRED_DIR / 'lstm_5day_forecasts.csv',
    ]
    
    all_forecasts = []
    for fpath in forecast_files:
        if fpath.exists():
            df = pd.read_csv(fpath)
            all_forecasts.append(df)
    
    if all_forecasts:
        forecasts_df = pd.concat(all_forecasts, ignore_index=True)
        # Average forecasts by ticker and day
        ensemble_fc = forecasts_df.groupby(['Ticker', 'Date'])['Forecast'].mean().reset_index()
        ensemble_fc['Date'] = pd.to_datetime(ensemble_fc['Date'])
        ensemble_fc = ensemble_fc.sort_values(['Ticker', 'Date'])
    else:
        print('WARNING: No forecast files found')
        ensemble_fc = pd.DataFrame()
    
    # ── Load actual prices for directional signals ────────────
    from src.data.preprocessor import SELECTED as tickers_all
    
    # ── Generate trading signals ──────────────────────────────
    signals = []
    
    for _, port_row in portfolio.iterrows():
        ticker = port_row['Ticker']
        current_price = port_row['Current_Price_₹']
        shares = port_row['Shares']
        weight = port_row['Actual_Weight_%']
        
        # Get forecast for this ticker
        tc_fc = ensemble_fc[ensemble_fc['Ticker'] == ticker].sort_values('Date')
        
        if not tc_fc.empty:
            day1_forecast = tc_fc.iloc[0]['Forecast'] if len(tc_fc) > 0 else current_price
            day5_forecast = tc_fc.iloc[-1]['Forecast'] if len(tc_fc) > 0 else current_price
            expected_move_5d = (day5_forecast - current_price) / current_price * 100
            direction = 'BUY' if expected_move_5d > 0.5 else 'SELL' if expected_move_5d < -0.5 else 'HOLD'
        else:
            day1_forecast = current_price
            day5_forecast = current_price
            expected_move_5d = 0.0
            direction = 'HOLD'
        
        signals.append({
            'Order_ID': f'{ticker}_{datetime.now().strftime("%Y%m%d")}',
            'Ticker': ticker,
            'Company': NAMES.get(ticker, ticker),
            'Signal': direction,
            'Current_Price_₹': round(current_price, 2),
            'Shares_to_Trade': abs(shares),
            'Action': 'BUY' if direction == 'BUY' else 'SELL' if direction == 'SELL' else 'HOLD',
            'Portfolio_Weight_%': weight,
            'Day1_Forecast_₹': round(day1_forecast, 2),
            'Day5_Forecast_₹': round(day5_forecast, 2),
            'Expected_Move_5D_%': round(expected_move_5d, 2),
            'Trade_Value_₹': round(shares * current_price, 0),
            'Confidence': 'HIGH' if abs(expected_move_5d) > 2.0 else 'MEDIUM' if abs(expected_move_5d) > 0.5 else 'LOW',
            'Timestamp_UTC': datetime.utcnow().isoformat(),
        })
    
    signals_df = pd.DataFrame(signals)
    
    # ── Day 1 Execution Instructions ──────────────────────────
    buy_signals = signals_df[signals_df['Action'] == 'BUY'].copy()
    sell_signals = signals_df[signals_df['Action'] == 'SELL'].copy()
    
    print('\n' + '='*80)
    print('DAY 1 — EXECUTION INSTRUCTIONS')
    print('='*80)
    
    day1_instructions = {
        'Execution_Date': (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d'),
        'Execution_Type': 'INTRADAY',
        'Buy_Orders': len(buy_signals),
        'Sell_Orders': len(sell_signals),
        'Hold_Positions': len(signals_df[signals_df['Action'] == 'HOLD']),
        'Total_Buy_Value_₹': round(buy_signals['Trade_Value_₹'].sum(), 0),
        'Total_Sell_Value_₹': round(sell_signals['Trade_Value_₹'].sum(), 0),
        'Net_Cash_Required_₹': round(buy_signals['Trade_Value_₹'].sum() - sell_signals['Trade_Value_₹'].sum(), 0),
    }
    
    print('\nExecution Summary:')
    for key, val in day1_instructions.items():
        print(f'  {key}: {val}')
    
    if not buy_signals.empty:
        print('\nBUY ORDERS:')
        print(buy_signals[['Ticker', 'Shares_to_Trade', 'Current_Price_₹', 'Trade_Value_₹', 'Confidence']].to_string(index=False))
    
    if not sell_signals.empty:
        print('\nSELL ORDERS:')
        print(sell_signals[['Ticker', 'Shares_to_Trade', 'Current_Price_₹', 'Trade_Value_₹', 'Confidence']].to_string(index=False))
    
    # ── Day 2 Tracking Instructions ──────────────────────────
    print('\n' + '='*80)
    print('DAY 2 — TRACKING & REVIEW INSTRUCTIONS')
    print('='*80)
    
    day2_tracking = {
        'Review_Time_IST': '15:30 (Market Close)',
        'Key_Metrics_To_Monitor': [
            'Actual price movements vs Day1 forecasts',
            'Directional accuracy of ensemble model',
            'Portfolio P&L attribution by stock',
            'Volatility realization vs GARCH forecast',
        ],
        'Rebalancing_Trigger': 'If any position moves >5% from target weight',
        'Stop_Loss_Trigger': 'If any position moves <-3% from entry',
        'Profit_Target': 'If any position moves >3% from entry (optional exit)',
    }
    
    print('\nDay 2 Tracking Schedule:')
    for key, val in day2_tracking.items():
        if isinstance(val, list):
            print(f'  {key}:')
            for item in val:
                print(f'    - {item}')
        else:
            print(f'  {key}: {val}')
    
    # ── Save StockGro-formatted outputs ───────────────────────
    signals_df.to_csv(PORT_DIR / 'stockgro_trade_instructions.csv', index=False)
    
    # Save execution plan as JSON for API integration
    import json
    day1_plan = {
        'execution_date': day1_instructions['Execution_Date'],
        'buy_orders': buy_signals[['Ticker', 'Shares_to_Trade', 'Current_Price_₹']].to_dict('records'),
        'sell_orders': sell_signals[['Ticker', 'Shares_to_Trade', 'Current_Price_₹']].to_dict('records'),
        'summary': day1_instructions,
    }
    
    with open(PORT_DIR / 'day1_execution_plan.json', 'w') as f:
        json.dump(day1_plan, f, indent=2)
    
    print(f'\n✅ StockGro signals generated. Saved:')
    print(f'   - {PORT_DIR / "stockgro_trade_instructions.csv"}')
    print(f'   - {PORT_DIR / "day1_execution_plan.json"}')
    
    print(f'\nReady for Day 1 execution at market open.')
    print(f'Next review scheduled for Day 2 close (15:30 IST).')